# Zimbabwe Used Vehicle Market Analysis
**Author:** Tatenda Alistair Gonzo  
**Snapshot:** 5 November 2026

This notebook performs an initial exploratory analysis of publicly advertised used-vehicle listings in Harare. The dataset is a small convenience sample and must not be treated as representative of the entire Zimbabwean market.

## 1. Import libraries and load the data
Run this notebook in Google Colab. Upload `used_vehicles.csv` when prompted.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()
csv_name = next(iter(uploaded))
vehicles = pd.read_csv(csv_name)

print(f"Loaded {len(vehicles)} listings and {len(vehicles.columns)} columns.")
vehicles.head()

## 2. Inspect the dataset

In [ ]:
print("Shape:", vehicles.shape)
print("\\nColumn data types:")
print(vehicles.dtypes)
print("\\nMissing values by column:")
display(vehicles.isna().sum().sort_values(ascending=False))

## 3. Prepare fields for analysis
We preserve the original columns and create calculated fields. Missing values are not imputed.

In [ ]:
vehicles["year"] = pd.to_numeric(vehicles["year"], errors="coerce")
vehicles["price"] = pd.to_numeric(vehicles["price"], errors="coerce")
vehicles["mileage_km"] = pd.to_numeric(vehicles["mileage_km"], errors="coerce")
vehicles["date_collected"] = pd.to_datetime(vehicles["date_collected"], errors="coerce")
vehicles["vehicle_age_at_snapshot"] = 2026 - vehicles["year"]

# Keep USD listings for price analysis
usd_vehicles = vehicles[vehicles["currency"].str.upper() == "USD"].copy()

print("USD listings:", len(usd_vehicles))
print("Duplicate listing IDs:", vehicles["listing_id"].duplicated().sum())

## 4. Descriptive statistics

In [ ]:
display(usd_vehicles[["price", "year", "mileage_km"]].describe().round(2))
print("\\nListings by make:")
display(vehicles["make"].value_counts())

## 5. Asking-price distribution

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(usd_vehicles["price"].dropna(), bins=8, edgecolor="black")
plt.title("Distribution of Advertised Asking Prices")
plt.xlabel("Asking price (USD)")
plt.ylabel("Number of listings")
plt.tight_layout()
plt.show()

## 6. Asking price by vehicle year

In [ ]:
price_by_year = usd_vehicles.groupby("year")["price"].median().sort_index()
plt.figure(figsize=(9, 5))
price_by_year.plot(kind="bar")
plt.title("Median Asking Price by Model Year")
plt.xlabel("Model year")
plt.ylabel("Median asking price (USD)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Mileage and asking price

In [ ]:
mileage_price = usd_vehicles.dropna(subset=["mileage_km", "price"])
plt.figure(figsize=(9, 5))
plt.scatter(mileage_price["mileage_km"], mileage_price["price"], alpha=0.8)
plt.title("Mileage vs Advertised Asking Price")
plt.xlabel("Mileage (km)")
plt.ylabel("Asking price (USD)")
plt.tight_layout()
plt.show()

if len(mileage_price) >= 3:
    print("Sample correlation (mileage vs price):",
          round(mileage_price["mileage_km"].corr(mileage_price["price"]), 3))

## 8. Transmission and fuel type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vehicles["transmission"].value_counts(dropna=False).plot(kind="bar", ax=axes[0], title="Listings by Transmission")
vehicles["fuel_type"].value_counts(dropna=False).plot(kind="bar", ax=axes[1], title="Listings by Fuel Type")
axes[0].set_xlabel("Transmission")
axes[1].set_xlabel("Fuel type")
axes[0].set_ylabel("Number of listings")
axes[1].set_ylabel("Number of listings")
plt.tight_layout()
plt.show()

## 9. Data-quality review

In [ ]:
quality = pd.DataFrame({
    "missing_count": vehicles.isna().sum(),
    "missing_percent": (vehicles.isna().mean() * 100).round(1)
}).sort_values("missing_percent", ascending=False)
display(quality)

# Potential duplicates across listing sources: review manually; do not delete automatically.
potential_duplicates = vehicles.duplicated(
    subset=["make", "model", "year", "price", "mileage_km"], keep=False
)
display(vehicles.loc[potential_duplicates, ["listing_id", "make", "model", "year", "price", "mileage_km", "source_name"]])

## 10. Interpretation and limitations
This is an exploratory convenience sample, dominated by Toyota Axio listings. Report statistics as sample-level observations only. Do not claim that these results represent all Zimbabwe used-car listings or actual sale prices. Expand and verify the dataset before drawing broader conclusions.

In [ ]:
print(f"Sample size: {len(vehicles)}")
print(f"Median asking price (USD): ${usd_vehicles['price'].median():,.0f}")
print(f"Median mileage (km): {usd_vehicles['mileage_km'].median():,.0f}")
print("Most common make:", vehicles["make"].mode().iat[0] if not vehicles["make"].mode().empty else "N/A")